# OCR Model Comparison: EasyOCR, Tesseract, Surya

This notebook initializes three OCR models with Romanian language support and demonstrates how to run them on batches of images.

## Prerequisites
Ensure you have run `setup.sh` to install system dependencies (Tesseract + Romanian lang) and Python packages.

In [1]:
import os
import glob
import time
import cv2
import torch
import easyocr
import pytesseract
from PIL import Image
import pandas as pd
from rapidfuzz.distance import Levenshtein, JaroWinkler
from jiwer import cer, wer

# Check Tesseract version and language
try:
    print(f"Tesseract Version: {pytesseract.get_tesseract_version()}")
    print(f"Available Tesseract Languages: {pytesseract.get_languages(config='')}")
except Exception as e:
    print("Error finding Tesseract. Make sure it is installed and in your PATH.")
    print(e)

# Check CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Tesseract Version: 5.3.4
Available Tesseract Languages: ['eng', 'osd', 'ron']
Using device: cuda


## 1. Initialize Models

In [2]:
# --- EasyOCR Initialization ---
# 'ro' for Romanian, 'en' for English (good fallback)
easyocr_reader = easyocr.Reader(['ro', 'en'], gpu=(device == 'cuda'))
print("EasyOCR initialized.")

EasyOCR initialized.


In [3]:
# --- Tesseract Initialization ---
# To initialize tesseract, we only need to set the configuration string
tess_config = r'--oem 3 --psm 3 -l ron'
print(f"Tesseract config set to: {tess_config}")

Tesseract config set to: --oem 3 --psm 3 -l ron


In [4]:

# --- Surya Initialization ---
from surya.detection import DetectionPredictor
from surya.recognition import RecognitionPredictor
from surya.foundation import FoundationPredictor
from surya.common.surya.schema import TaskNames

# Initialize predictors
foundation_predictor = FoundationPredictor()
det_predictor = DetectionPredictor()
rec_predictor = RecognitionPredictor(foundation_predictor)

print("Surya initialized.")

Surya initialized.


## 2. Helper Functions

In [5]:
def load_images(source):
    """
    Loads images from a directory path or a list of file paths.
    Returns a list of (path, PIL.Image) tuples.
    """
    image_paths = []
    if isinstance(source, str):
        if os.path.isdir(source):
            # Common image extensions
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.pdf']
            for ext in extensions:
                image_paths.extend(glob.glob(os.path.join(source, ext)))
        elif os.path.isfile(source):
            image_paths = [source]
    elif isinstance(source, list):
        image_paths = source
    
    images = []
    for p in image_paths:
        try:
            img = Image.open(p).convert('RGB')
            images.append((p, img))
        except Exception as e:
            print(f"Could not load image {p}: {e}")
            
    return images

## 3. Batch Processing Functions

In [6]:
def run_easyocr_batch(images):
    """
    Runs EasyOCR on a list of (path, PIL.Image) tuples.
    """
    results = {}
    print(f"Running EasyOCR on {len(images)} images...")
    start_time = time.time()
    
    for path, img in images:
        # EasyOCR accepts file path, numpy array, or bytes
        # We pass the path directly or convert PIL to numpy
        import numpy as np
        img_np = np.array(img)
        
        # detail=0 returns just the text list
        text = easyocr_reader.readtext(img_np, detail=0)
        results[path] = " ".join(text)
        
    end_time = time.time()
    print(f"EasyOCR finished in {end_time - start_time:.2f}s")
    return results

def run_tesseract_batch(images):
    """
    Runs Tesseract on a list of (path, PIL.Image) tuples.
    """
    results = {}
    print(f"Running Tesseract on {len(images)} images...")
    start_time = time.time()
    
    for path, img in images:
        text = pytesseract.image_to_string(img, lang='ron')
        results[path] = text.strip()
        
    end_time = time.time()
    print(f"Tesseract finished in {end_time - start_time:.2f}s")
    return results

def run_surya_batch(images):
    """
    Runs Surya on a list of (path, PIL.Image) tuples.
    Surya has native batch support.
    """
    results = {}
    print(f"Running Surya on {len(images)} images...")
    start_time = time.time()
    
    pil_images = [img for _, img in images]
    paths = [p for p, _ in images]
    
    # Prepare task names
    task_names = [TaskNames.ocr_with_boxes] * len(pil_images)

    # Run prediction
    predictions = rec_predictor(
        pil_images,
        task_names=task_names,
        det_predictor=det_predictor
    )
    
    for path, pred in zip(paths, predictions):
        # pred is an OCRResult object, we extract text lines
        text_lines = [l.text for l in pred.text_lines]
        results[path] = " ".join(text_lines)
        
    end_time = time.time()
    print(f"Surya finished in {end_time - start_time:.2f}s")
    return results

## 4. Metrics Implementation
We define functions to calculate Levenshtein Distance, Jaro-Winkler Similarity, CER, and WER.

In [9]:
def calculate_metrics(ground_truth, hypothesis):
    """
    Calculates OCR metrics between ground truth and hypothesis text.
    Returns a dictionary with metrics.
    """
    if not ground_truth or not hypothesis:
        return {
            "Levenshtein": None,
            "Jaro-Winkler": None,
            "CER": None,
            "WER": None
        }
    
    # Levenshtein Distance (lower is better)
    lev_dist = Levenshtein.distance(ground_truth, hypothesis)
    
    # Jaro-Winkler Similarity (higher is better, 0 to 1)
    jw_sim = JaroWinkler.similarity(ground_truth, hypothesis)
    
    # Character Error Rate (lower is better)
    # We wrap in try-except because jiwer might fail on empty strings
    try:
        cer_score = cer(ground_truth, hypothesis)
    except:
        cer_score = None
        
    # Word Error Rate (lower is better)
    try:
        wer_score = wer(ground_truth, hypothesis)
    except:
        wer_score = None
        
    return {
        "Levenshtein": lev_dist,
        "Jaro-Winkler": jw_sim,
        "CER": cer_score,
        "WER": wer_score
    }

## 5. Execution & Comparison
Set `image_source` and optionally provide `ground_truth_map`.

In [10]:
# PDF support helper: convert PDF pages to PIL Images (best-effort)
try:
    from pdf2image import convert_from_path
    _pdf2image_available = True
    _pdf2image_error = None
except Exception as _e:
    _pdf2image_available = False
    _pdf2image_error = str(_e)

def load_pdf_as_images(pdf_path, dpi=300):
    """Convert a PDF file into a list of PIL.Image pages using pdf2image.
    Returns list of PIL.Image objects. Raises RuntimeError with instructions
    if pdf2image or poppler are not available.
    """
    if not _pdf2image_available:
        raise RuntimeError("pdf2image is not installed or failed to import: " + (_pdf2image_error or "unknown"))
    try:
        pages = convert_from_path(pdf_path, dpi=dpi)
        return pages
    except Exception as e:
        raise RuntimeError(
            f"Failed to convert PDF to images (poppler may be missing or convert_from_path failed): {e}\n"
            "On Linux install poppler (apt): sudo apt-get install -y poppler-utils"
        )

# New loader that supports image files and PDF pages
from glob import glob
def load_images_with_pdfs(source, pdf_dpi=300):
    """Load images from directory/list/file and convert PDF pages to images.
    Returns list of (path_identifier, PIL.Image) tuples. For PDF pages the identifier
    is 'original.pdf::page_N'.
    """
    image_entries = []
    image_paths = []

    if isinstance(source, str):
        if os.path.isdir(source):
            extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tiff', '*.pdf']
            for ext in extensions:
                image_paths.extend(sorted(glob(os.path.join(source, ext))))
        elif os.path.isfile(source):
            image_paths = [source]
    elif isinstance(source, list):
        image_paths = source

    for p in image_paths:
        try:
            ext = os.path.splitext(p)[1].lower()
            if ext == '.pdf':
                # convert PDF pages to images
                try:
                    pages = load_images(p, dpi=pdf_dpi)
                    for i, page in enumerate(pages, start=1):
                        id_path = f"{p}::page_{i}"
                        image_entries.append((id_path, page))
                except Exception as e:
                    print(f"Could not convert PDF {p}: {e}")
            else:
                img = Image.open(p).convert('RGB')
                image_entries.append((p, img))
        except Exception as e:
            print(f"Could not load image/file {p}: {e}")

    return image_entries

# Quick status print
print('PDF helper loaded. pdf2image available =', _pdf2image_available)
if not _pdf2image_available:
    print("\nTo enable PDF -> image conversion install:")
    print("  pip install pdf2image")
    print("\nAnd install system dependency poppler on Linux:")
    print("  sudo apt-get update && sudo apt-get install -y poppler-utils")
    print("\nOr on other OS follow poppler installation instructions.")

PDF helper loaded. pdf2image available = True


In [11]:

image_source = 'TestImages' 
# "1.png": "Un popor care nu își cunoaște istoria este ca un copil care nu își cunoaște părinții",
#     "6.png": "Un gînd frumos poate fi ca o biserică în care omul își odihnește sufletul",
#     "ArhivaOperativa.png": "MINISTERUL AFACERILOR INTERNE DOS. NR.40010 VOL. Nr.125 ARHIVA OPERATIVA DOSAR Anchetă",
#     "image.png": "1. Descriere afacere:a. Denumire: ConveConnect b. Localizarea afacerii: Strada Dumitru Florescu nr 32 c. Descrierea produsului: Firma care se ocupa cu proiectarea si vanzarea convertoarelor de protocol pentru dispozitive de automatizare a cladirilor. d. Componenta de inovare: Convertoarele existente pe piata sunt costisitoare si greu de procurat rapid sau sunt de calitate inferioara, mai ales la nivel de software de configurare. Noi putem dezvolta propriul design hardware si software, urmand sa externalizam productia. Vom crea o interfata intuitiva si usor de utilizat. e. Forma juridică: SRL f. Domeniu de activitate principal: 2790 Fabricarea altor echipamente electrice g. Tipul afacerii: Neplatitoare de TVA ? h. Valoare capital social: 200 RON ? i. Prezentare administrator, angajati: i. Administrator: Bogdan Paul Prisacaru 1. Rol: administrator unic 2. Nr angajati: 1 3. Descriere: Asigura management-ul strategic si operational al companiei. Pe langa sarcinile administrative, legale si financiare, se va ocupa direct de dezvoltarea afacerii 4. Responsabilitati cheie: administrarea generala a firmei Identificarea pietei tinta, stabilirea preturilor Gestionarea fluxului de numerar si a investitiilor ii. Angajat: Proiectant Hardware & Software (Inginer Embedded Systems) 1. Rol: pilonul tehnic al companiei, responsabil pentru transpunerea viziunii in produse functionale. Are in vedere componenta de inovare (software + design propriu) 2. Nr angajati: 3 3. Responsabilitati cheie: Proiectare hardware: realizarea schemelor electronice si proiectarea PCB-urilor (Printed Circuit Board) pentru convertoare Dezvoltare Firmware: Programarea microcontrolerelor (in limbaje de tip C/C++) care ruleaza pe dispozitive, implementand protocoale de comunicatie (ex. BACnet, Modbus, KNX, MBus) Testare si validare: asigura calitatea si fiabilitatea produselor inainte de lansarea pe piata Suport tehnic: ofera suport de nivel inalt integratorilor care intampina probleme",
#     "masinaDeScris.png": "notă informativă bătută la mașină",
# --- GROUND TRUTH SETUP ---
ground_truth_map = {
    "Poza.pdf::page_1": "MINISTERUL COMERȚULUI EXTERIOR ȘI COOPERĂRII ECONOMICE INTERNAȚIONALE INSTITUTUL CENTRAL DE CERCETĂRI ECONOMICE INSTITUTUL DE ECONOMIE MONDIALĂ conjuctura economiei mondiale",
    "Poza.pdf::page_2": "PARTICIPAREA ACTIVĂ A ROMÂNIEI LA VIAȚA ECONOMICĂ INTERNAȚIONALĂ Ion PĂȚAN ministrul comerțului exterior și cooperării economice internaționale viceprim-ministru al guvernului Anul 1976 - primul an din actualul plan cincinal, al afirmarii revoluției technico-științifice - a marcat o prezență activă a țării noastre în viața economică internațională, atît sub aspectul angrenării mai profunde în cir-cuitul economic mondial, cît și prin amplifi-carea acțiunilor concrete consacrate soluționării marilor probleme economice care confruntă lumea contemporană. Intensificarea participării României socialiste la diviziunea mondială a muncii și, în acest cadru, dezvoltarea continuă a schimburilor și cooperării economice cu țările socialiste, cu țările în curs de dezvoltare, cu toate țările lumii fără deosebire de sistemul politico-social, pe baza principiilor progresiste ale relațiilor dintre state, precum și activitatea internațională des-fășurată pe multiple planuri în vederea înfăp-tuirii unei noi ordini economice mondiale dau expresie traducerii consecvente în viață a mă-rețelor obiective adoptate de Congresul al XI-lea al Partidului Comunist Român. Așa cum sublinia tovarășul Nicolae Ceaușescu în Raportul prezentat la Congresul al XI-lea al P.C.R., \"România va participa și în viitor, în mod activ, la diviziunea internațională a muncii, la schimbul de valori materiale și spirituale cu toate statele, fără deo-sebire de orînduirea socială. Considerăm că aceasta constituie o necesitate obiectivă pentru progresul economico-social al fiecărei țări - deci și României - precum și pentru reali-zarea unei noi ordini economice internaționale, pentru cauza colaborării și păcii în lume\".Transpunerea în fapt a orientărilor stabilite, contribuția mereu sporită a schimburilor comer-ciale și cooperării externe la dezvoltarea multi-laterală și în ritm susținut a economiei, apre-cierea largă și prestigiul tot mai mare de care România se bucură în lume ca partener eco-nomic și ca factor proeminent în lupta pentru progres, prosperitate și pace pe plan interna-țional au confirmat pe deplin justețea și rea-lismul politicii interne și externe a partidului și statului nostru, fundamentată în document-ele programatice ale Congresului al XI-lea al P.C.R.  În procesul de amplificare a participării României la viața economică internațională un rol esențial îl are activitatea multilaterală, inter-nă și internațională, desfășurată de secretarul ge-neral al partidului președintele republicii, tovarășul Nicolae Ceaușescu. Exigențele impri-mate de conducătorul partidului și statului pen-tru ridicarea continuă, cantitativă și calitativă a producției, îndrumarea și conducerea concretă, zi de zi, a activității în mod hotărîtor, la creșterea aportului sectorului extern al economiei în dez-voltarea generală a patriei. Acordurile economice și convențiile de cooperare încheiate cu un număr mare de state - rezultat direct al con-vorbirilor și înțelegerilor la nivel înalt - au impulsionat puternic extinderea relațiilor eco-nomice externe ale țării noastre. De cea mai mare însemnătate este contribuția prestigioasă, teoretică și practică, a tovarășului Nicolae Ceaușescu la afirmarea în viață a principiilor noi ale relațiilor între state, a unei noi ordini economice și politice internaționale, obiective cărora secretarul general al partidului le dedică o activitate intensă, neobosită, amplu remarcată și apreciată în lume.  Acționînd consecvent pentru extinderea schimnburilor sale economice externe, România devine un participant tot mai activ la circuitul economic mondial. Volumul total al comerțului exterior românesc a crescut continuu, fiind în 1975 de peste 19 ori mai ridicat decît cel din 1950 și de 6,5 ori mai mare decît cel din 1960. Semnificativ pentru angrenarea țării în circui-tul economic mondial este faptul că, în ultimii ani, schimburile economice externe s-au dezvol-tat într-un ritm mai înalt decît cel înregistrat de alți indicatori sintetici, ceea ce se înscrie în",
    "Poza.pdf::page_3": "tendințele calitative de pe arena economică in-ternațională. Astfel, în perioada 1971-1975, rit-mul mediu anual de creștere al comerțului exte-rior al țării noastre a fost de 18,3 la suta, mai rapid în comparație atît cu ritmul de creștere al produsului social (10,6 la sută pe an), venitu-lui național (11,3 la sută pe an), cît și cu cel al producției industriale (13 la suta pe an). Creșterea accelerată a relațiilor economice externe ale României a condus la sporirea ponderii activității de comerț exterior și cooperare cu alte țări la circa o pătrime în volumul total al producției. Pe această bază, se poate spune că fiecare al patrulea muncitor din producție este antrenat în activități care privesc exportul, importul sau cooperarea economică. Totodată, ca urmare a unei asemenea dezvoltări a schim-burilor economice cu alte state, țara noastră și-a majorat ponderea în ansamblul comerțului mon-dial de la 0,37 la sută în 1950 la 0,60 la sută în 1975. Succesele remarcabile obținute de poporul român în toate sectoarele economiei naționale în primul an al cincinaulului 1976-1980 , precum și măsurile luate pentru perfecționarea meca-nismului și sistemului de lucru în activitatea de comerț exterior și de cooperare internațională au creat condiții favorabile pentru o nouă dez-voltare a relațiilor economice externe ale țării noastre, pentru îmbunătățirea pe mai departe a structurii schimburilor comerciale și a lărgirii ariei lor geografice. În anul 1976 volumul comerțului exterior a crescut cu 14,5 la sută față de realizările anu-lui 1975, din care exportul cu 14,9 la sută, iar importul cu 14,1 la sută, balanța comercială externă fiind activă. Este, de asemenea, de remarcat faptul că sporirea volumului comerțu-lui exterior și cooperării economice al țării noastre s-a realizat printr-o diversificare mai largă a piețelor, România întreținând actial-mente relații economice cu 140 de țări ale lumii, de peste patru ori mai mult decît în primii ani postbelici. Așa cum se cunoaște, în ultimii ani activi-tatea noastră de comerț exterior și de cooperare s-a desfășurat în condițiile unor sensibile dere-glari pe piețele internaționale de mărfuri și financiare, ale existenței unei adînci crize, apre-ciată drept cea mai puternică din anii postbelici în cadrul economiei țărilor capitaliste. Con-junctura mondială în 1976, deși a marcat o anumită revenire față de anii trecuți, a resimțit consecințele acestei crize, indicatorii economici evidețiind o slăbire a tendințelor de relansare economică în semestrul al doilea al anului, com-parativ cu primul semestru, ceea ce s-a reflec-tat și asupra comerțului internațional. Cu toate acestea, relațiile economice externe ale țării noastre, întreaga economie națională au con-tinuat să se dezvolte în ritmuri înalte, punînd puternic în evidență superioritatea orînduirii noastre socialiste. În cadrul problematicii dezvoltării comerțu-lui exterior, un loc central a revenit, în con-tinuare, preocupărilor pentru creșterea ponderii în exportul românesc a produselor ce înglo-beaza o cantitate cît mai mare de muncă com-plexă, de tehnicitate și cercetare, pentru a asi-gura valorificarea superioară a resurselor materiale și de muncă, a potențialului indus-trial și tehnologic al țării. Produsele care răs-pund dezideratelor de eficiență economică spo-rită - mașini și utilake, produse chimice și bunuri industriale de larg consum - au deținut în 1976 peste 50 la sută din totalul exportului. În cadrul fiecărei ramuri s-au depus eforturi pentru promovarea exportului produselor care încorporează un aport cît mai mare de creație proprie, pentru alinierea parametrilor tehnici și de calitate la cerințele piețelor externe. Printre produsele noi introduse la export în cursul anului 1976 se numără autoturismele de teren ARO 241,242 și 244, autobuzele ROMAN 109 RD și 112 UD, locomotivele Diesel electrice de 3 500 CP, tractorul multifuncțional TIH-445 și tractorul pe șenile S-1 800 IF, noi tipuri de strunguri automate și freze, centrale telefonice rurale, televizoarele cu circuite integrate, mași-nile de facturat FC 16 și 64, calculatoarele de birou Felix CE 211 și 212 T, cauciucul poliizo-prenic, numeroase modele noi de mobilă, con-fecții, textile etc. Nivelul atins de comerțul nostru exterior ne obligă la intensificarea eforturilor în direcția lărgirii și îmbunătățirii eficienței participării la circuitul economic mondial. Mai avem încă mult de înfăptuit în acest sens, luînd în con-siderare chiar și numai faptul că, în timp ce România realizeaza circa 1,50 la sută din pro-ducția industrială mondială, ponderea ei în comerțul mondial este mai mică. În ceea ce pri-vește structura schimburilor, exportul în țările capitaliste dezvoltate este compus încă în mare măsură din mărfuri de bază și semifabricate, datorită, în mare parte, și restricțiilor mai vechi și noi impuse de aceste țări la importul de pro-duse finite.",
    "Poza.pdf::page_4": "",
}

# Create dummy image if needed
if not os.path.exists(image_source):
    os.makedirs(image_source)
    from PIL import Image, ImageDraw
    dummy_img = Image.new('RGB', (400, 100), color = (255, 255, 255))
    d = ImageDraw.Draw(dummy_img)
    d.text((10,10), "Acesta este un test OCR.", fill=(0,0,0))
    dummy_img.save(os.path.join(image_source, 'test1.png'))
    print(f"Created dummy image in {image_source}")

batch_images = load_images_with_pdfs(image_source)
print(f"Loaded {len(batch_images)} images.")

if batch_images:
    # Run Models
    easy_results = run_easyocr_batch(batch_images)
    tess_results = run_tesseract_batch(batch_images)
    surya_results = run_surya_batch(batch_images)
    
    # Compile Results into DataFrame
    data = []
    
    for path, _ in batch_images:
        filename = os.path.basename(path)
        gt = ground_truth_map.get(filename, "")
        
        # Get predictions
        easy_pred = easy_results.get(path, "")
        tess_pred = tess_results.get(path, "")
        surya_pred = surya_results.get(path, "")
        
        # Calculate Metrics (comparing against Ground Truth)
        # We calculate metrics for each model
        easy_metrics = calculate_metrics(gt, easy_pred)
        tess_metrics = calculate_metrics(gt, tess_pred)
        surya_metrics = calculate_metrics(gt, surya_pred)
        
        row = {
            "Image": filename,
            "Ground Truth": gt,
            
            "EasyOCR Text": easy_pred,
            "EasyOCR Lev": easy_metrics["Levenshtein"],
            "EasyOCR JW": easy_metrics["Jaro-Winkler"],
            "EasyOCR CER": easy_metrics["CER"],
            "EasyOCR WER": easy_metrics["WER"],
            
            "Tesseract Text": tess_pred,
            "Tesseract Lev": tess_metrics["Levenshtein"],
            "Tesseract JW": tess_metrics["Jaro-Winkler"],
            "Tesseract CER": tess_metrics["CER"],
            "Tesseract WER": tess_metrics["WER"],
            
            "Surya Text": surya_pred,
            "Surya Lev": surya_metrics["Levenshtein"],
            "Surya JW": surya_metrics["Jaro-Winkler"],
            "Surya CER": surya_metrics["CER"],
            "Surya WER": surya_metrics["WER"],
        }
        data.append(row)
        
    df = pd.DataFrame(data)
    
    # Save to CSV
    df.to_csv("ocr_benchmark_results.csv", index=False)
    print("Results saved to ocr_benchmark_results.csv")
    print("\n" + "="*80)

Loaded 10 images.
Running EasyOCR on 10 images...
EasyOCR finished in 11.06s
Running Tesseract on 10 images...
Tesseract finished in 8.12s
Running Surya on 10 images...


Recognizing Text: 100%|██████████| 366/366 [00:25<00:00, 14.50it/s]


Surya finished in 28.77s
Results saved to ocr_benchmark_results.csv



## 6. Display Results
We'll display the results in multiple tables for better readability.

In [12]:
# Configure pandas to show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', None)

print("\nSFULL COMPARISON TABLE")
print("=" * 150)
display(df)

print("\n\nMETRICS SUMMARY (CER - Character Error Rate)")
print("=" * 80)
print("Lower is better. 0.0 = perfect match, 1.0 = 100% error\n")
metrics_df = df[['Image', 'EasyOCR CER', 'Tesseract CER', 'Surya CER']].copy()
display(metrics_df)

print("\n\nMETRICS SUMMARY (Jaro-Winkler Similarity)")
print("=" * 80)
print("Higher is better. 1.0 = perfect match, 0.0 = no similarity\n")
jw_df = df[['Image', 'EasyOCR JW', 'Tesseract JW', 'Surya JW']].copy()
display(jw_df)

print("\n\nMETRICS SUMMARY (Levenshtein Distance)")
print("=" * 80)
print("Lower is better. 0 = perfect match, higher = more edits needed\n")
lev_df = df[['Image', 'EasyOCR Lev', 'Tesseract Lev', 'Surya Lev']].copy()
display(lev_df)

# Calculate average metrics (excluding None values)
print("\n\nAVERAGE METRICS ACROSS ALL IMAGES")
print("=" * 80)
avg_metrics = {
    'Model': ['EasyOCR', 'Tesseract', 'Surya'],
    'Avg CER': [
        df['EasyOCR CER'].mean(),
        df['Tesseract CER'].mean(),
        df['Surya CER'].mean()
    ],
    'Avg Jaro-Winkler': [
        df['EasyOCR JW'].mean(),
        df['Tesseract JW'].mean(),
        df['Surya JW'].mean()
    ],
    'Avg Levenshtein': [
        df['EasyOCR Lev'].mean(),
        df['Tesseract Lev'].mean(),
        df['Surya Lev'].mean()
    ]
}
avg_df = pd.DataFrame(avg_metrics)
display(avg_df)


SFULL COMPARISON TABLE


,Image,Ground Truth,EasyOCR Text,EasyOCR Lev,EasyOCR JW,EasyOCR CER,EasyOCR WER,Tesseract Text,Tesseract Lev,Tesseract JW,Tesseract CER,Tesseract WER,Surya Text,Surya Lev,Surya JW,Surya CER,Surya WER
0,image.png,,"A nul 1976 primul an din actualul plan cincinal, al afirmării revolutiei tehnico stiinț...",None,None,None,None,"Anu 1976 — primul an din actualul, plan\nal. afirmării revoluţiei tehnico-\n\ncincinal,\nce — a ...",None,None,None,None,"nul 1976 — primul an din actualul plan cincinal, al afirmării revoluției tehnico- științifice — ...",None,None,None,None
1,page_2_header.png,,PHRTICIPAREA HCTIVĂ 'A- ROMANIEI LA VIATA 'ECONOIICĂ IIVTERNAȚIONHLĂ,None,None,None,None,OTV VP PIPI Pr pp Y Li\n\nu] |\n\n- PARTICIPAREA ACIIVĂ “A ROMÂNIEI\nLA VIATA: PCONOMICĂ INTERNA...,None,None,None,None,PARTICIPAREA ACTIVA A ROMÂNIEI VIATA ECONOMICĂ INTERNATIONALĂ,None,None,None,None
2,page_2_left.png,,lon ministrul con șl cooperării econ viceprIm-mInls F nul 1976 ~ primul an din actualul plan ci...,None,None,None,None,"; lon\n\nministrul con\n, şi cooperării econ\n„ viceprim-minis\nLi\nLS ” 4 A\nMI)\n\nA rul 1976 ...",None,None,None,None,lon ministrul con și cooperării econ viceprim-minis nul 1976.— primul an din actualul plan cinci...,None,None,None,None
3,page_2_right.png,,PĂTAN ertuluf exterior mice Internationale ru al guvernului lismul .politicii interne 'și exter...,None,None,None,None,", !\n\nPĂŢAN\nerțului exterior |\nmice internaţionale\n\nru al guvernului\n\n! b\n\n„i 4. ” S .\...",None,None,None,None,<b>PATAN</b> ertului exterior omice internaționale ru al guvernului lismul politicii interne și ...,None,None,None,None
4,page_3_header.png,,conJunciura economlef mondlale',None,None,None,None,"Lat)\n\ni Și ț Pa i\n\n* cinlunctura economia mândiale\n\n=)\n\n,\n\n.\n\nv,\nSid pu\n\nCI",None,None,None,None,Conjunctura economiei mondiale,None,None,None,None
5,page_3_left.png,,"tendințele calitative de , p arena.economică ternational . As'fel, In perioada 197121075,rit- mu...",None,None,None,None,"4 la fa . Li -\ntendinţele calitativ de, pă arena. economică în !\nternaţională, Askţel, în peri...",None,None,None,None,"tendințele calitative de pe arena economică in- ternațională. Astfel, în perioada 1971-1975, rit...",None,None,None,None
6,page_3_right.png,,"slăbire a tendințelor de relansare gonemtioad în semestrul' al doilea al anului, com- parativ Cl...",None,None,None,None,"!\n\n, oviăențiină, o slăbire a, tendinţelor de relansare\n Săonomică în semestrul! al doilea a...",None,None,None,None,"evidentiind, o släbire a tendintelor de relansare économică în semestrul'al doilea al anului, co...",None,None,None,None
7,page_4_header.png,,Participarea actlvă a Românlel Ja viafa economfcă 'Internatională,None,None,None,None,"| , y Pa ali IT PTT TI Popor mp ere | pe mere pre er\nbă 4 h ' ] , [i Li\n) d ț\n. . A 4 U eu . ...",None,None,None,None,"Mil. Vis. Participarea activă a României la viața economică internațională : 10% ` <i>T</i>, & 5...",None,None,None,None
8,page_4_left.png,,Sarcini deosebite stau în fata activitătii de comert exterior ;i cooperare economică inter- ...,None,None,None,None,ă\nSarcini deosebite stau în fața activităţii de\ncomerț exterior şi coopbrare economică înter-\...,None,None,None,None,Sarcini deosebite stau în fața activității de comert exterior și cooperare economică inter- nați...,None,None,None,None
9,page_4_right.png,,"In vederea dezvoltării cooperării, ân andl '976 Qu fost încheiate 38 de noi âcorduri }i ântelege...",None,None,None,None,"LEI FI ?\n\n' În vederea dezvoltării cooperării, în aratati 976 *\nau fost încheiate 38 de noi d...",None,None,None,None,"In vederea dezvoltării cooperării, în anul 1976 au fost încheiate 38 de noi acorduri și înțelege...",None,None,None,None




METRICS SUMMARY (CER - Character Error Rate)
Lower is better. 0.0 = perfect match, 1.0 = 100% error



,Image,EasyOCR CER,Tesseract CER,Surya CER
0,image.png,None,None,None
1,page_2_header.png,None,None,None
2,page_2_left.png,None,None,None
3,page_2_right.png,None,None,None
4,page_3_header.png,None,None,None
5,page_3_left.png,None,None,None
6,page_3_right.png,None,None,None
7,page_4_header.png,None,None,None
8,page_4_left.png,None,None,None
9,page_4_right.png,None,None,None




METRICS SUMMARY (Jaro-Winkler Similarity)
Higher is better. 1.0 = perfect match, 0.0 = no similarity



,Image,EasyOCR JW,Tesseract JW,Surya JW
0,image.png,None,None,None
1,page_2_header.png,None,None,None
2,page_2_left.png,None,None,None
3,page_2_right.png,None,None,None
4,page_3_header.png,None,None,None
5,page_3_left.png,None,None,None
6,page_3_right.png,None,None,None
7,page_4_header.png,None,None,None
8,page_4_left.png,None,None,None
9,page_4_right.png,None,None,None




METRICS SUMMARY (Levenshtein Distance)
Lower is better. 0 = perfect match, higher = more edits needed



,Image,EasyOCR Lev,Tesseract Lev,Surya Lev
0,image.png,None,None,None
1,page_2_header.png,None,None,None
2,page_2_left.png,None,None,None
3,page_2_right.png,None,None,None
4,page_3_header.png,None,None,None
5,page_3_left.png,None,None,None
6,page_3_right.png,None,None,None
7,page_4_header.png,None,None,None
8,page_4_left.png,None,None,None
9,page_4_right.png,None,None,None




AVERAGE METRICS ACROSS ALL IMAGES


,Model,Avg CER,Avg Jaro-Winkler,Avg Levenshtein
0,EasyOCR,NaN,NaN,NaN
1,Tesseract,NaN,NaN,NaN
2,Surya,NaN,NaN,NaN


In [13]:
import pandas as pd
import os

# Citeste rezultatele
df = pd.read_csv("ocr_benchmark_results.csv")

# Creaza folder
output_dir = "results_readable"
os.makedirs(output_dir, exist_ok=True)

def save_model_results(model_name, text_col, cer_col, wer_col):
    filename = os.path.join(output_dir, f"{model_name.lower()}_results.txt")
    
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(f"{'='*80}\n")
        f.write(f"  REZULTATE {model_name.upper()}\n")
        f.write(f"{'='*80}\n\n")
        
        for idx, row in df.iterrows():
            image_name = row["Image"]
            ground_truth = row["Ground Truth"] if pd.notna(row["Ground Truth"]) else "(lipseste)"
            ocr_text = row[text_col] if pd.notna(row[text_col]) else "(niciun output)"
            cer = row[cer_col] if pd.notna(row[cer_col]) else "N/A"
            wer = row[wer_col] if pd.notna(row[wer_col]) else "N/A"
            
            f.write(f"{'-'*80}\n")
            f.write(f"IMAGINE: {image_name}\n")
            f.write(f"{'-'*80}\n\n")
            
            f.write(f"METRICI:\n")
            f.write(f"   CER: {cer}\n")
            f.write(f"   WER: {wer}\n\n")
            
            f.write(f"GROUND TRUTH:\n")
            f.write(f"{'-'*40}\n")
            gt_lines = [ground_truth[i:i+80] for i in range(0, len(str(ground_truth)), 80)]
            for line in gt_lines:
                f.write(f"{line}\n")
            f.write(f"\n")
            
            f.write(f"OUTPUT {model_name.upper()}:\n")
            f.write(f"{'-'*40}\n")
            ocr_lines = [str(ocr_text)[i:i+80] for i in range(0, len(str(ocr_text)), 80)]
            for line in ocr_lines:
                f.write(f"{line}\n")
            f.write(f"\n\n")
        
        f.write(f"{'='*80}\n")
        f.write(f"  SFARSIT REZULTATE {model_name.upper()}\n")
        f.write(f"{'='*80}\n")
    
    print(f"Salvat: {filename}")

# Genereaza pentru fiecare model
save_model_results("EasyOCR", "EasyOCR Text", "EasyOCR CER", "EasyOCR WER")
save_model_results("Tesseract", "Tesseract Text", "Tesseract CER", "Tesseract WER")
save_model_results("Surya", "Surya Text", "Surya CER", "Surya WER")

with open(os.path.join(output_dir, "comparatie_metrici.txt"), 'w', encoding='utf-8') as f:
    f.write(f"{'='*80}\n")
    f.write(f"  COMPARATIE METRICI OCR\n")
    f.write(f"{'='*80}\n\n")
    
    for idx, row in df.iterrows():
        f.write(f"{row['Image']}\n")
        f.write(f"   {'-'*50}\n")
        f.write(f"   {'Model':<12} {'CER':<10} {'WER':<10}\n")
        f.write(f"   {'-'*50}\n")
        
        cer_easy = row['EasyOCR CER'] if pd.notna(row['EasyOCR CER']) else 'N/A'
        cer_tess = row['Tesseract CER'] if pd.notna(row['Tesseract CER']) else 'N/A'
        cer_surya = row['Surya CER'] if pd.notna(row['Surya CER']) else 'N/A'
        
        wer_easy = row['EasyOCR WER'] if pd.notna(row['EasyOCR WER']) else 'N/A'
        wer_tess = row['Tesseract WER'] if pd.notna(row['Tesseract WER']) else 'N/A'
        wer_surya = row['Surya WER'] if pd.notna(row['Surya WER']) else 'N/A'
        
        f.write(f"   {'EasyOCR':<12} {str(cer_easy):<10} {str(wer_easy):<10}\n")
        f.write(f"   {'Tesseract':<12} {str(cer_tess):<10} {str(wer_tess):<10}\n")
        f.write(f"   {'Surya':<12} {str(cer_surya):<10} {str(wer_surya):<10}\n")
        f.write(f"\n")

print(f"Salvat: {output_dir}/comparatie_metrici.txt")
print(f"\nToate rezultatele sunt in folderul: {output_dir}/")


Salvat: results_readable/easyocr_results.txt
Salvat: results_readable/tesseract_results.txt
Salvat: results_readable/surya_results.txt
Salvat: results_readable/comparatie_metrici.txt

Toate rezultatele sunt in folderul: results_readable/


In [14]:
import os

pdf_path = "TestImages/Poza.pdf" 
output_folder = "extracted_pages" 
dpi = 300 

os.makedirs(output_folder, exist_ok=True)

pages = load_pdf_as_images(pdf_path, dpi=dpi)

for i, page in enumerate(pages, start=1):
    output_path = os.path.join(output_folder, f"page_{i}.png")
    page.save(output_path, "PNG")
    print(f"Salvat: {output_path}")

print(f"\nTotal: {len(pages)} pagini salvate in {output_folder}/")

RuntimeError: Failed to convert PDF to images (poppler may be missing or convert_from_path failed): Unable to get page count.
I/O Error: Couldn't open file 'TestImages/Poza.pdf': No such file or directory.

On Linux install poppler (apt): sudo apt-get install -y poppler-utils